# Lab 08 : Pandas and Ocean-Acidification Buoys
### EES 3350/5350 · Python in Earth Science · Monday, October 19, 2026
**Due:** Sunday, October 25, at 11:59 pm, submitted by pushing to your Week 08 assignment repo.

The ocean has absorbed roughly a quarter of the carbon dioxide that humans have reseased by burning fossil fuels ([Friedlingstein et al., 2023](https://doi.org/10.5194/essd-15-5301-2023)). That's good news for the atmosphere and bad news for the seawater: dissolved CO$_2$ makes carbonic acid, and the average pH of the surface ocean has already dropped by about 0.1 since pre-industrial times ([IPCC, 2021](https://www.ipcc.ch/report/ar6/wg1/chapter/summary-for-policymakers/)). Lower pH makes it harder for corals, oysters, and plankton to build their shells and skeletons. This is **ocean acidification**.

**What's "normal" pH for seawater?** The pH scale runs from 0 to 14. Pure water is neutral at 7, lower values are acidic (lemon juice is about 2), and higher values are basic (baking soda in water is about 8 to 9). Seawater is slightly **basic**: today, the surface of the open ocean averages a pH of about **8.1**, down from about 8.2 before the industrial era (IPCC, 2001). Water near coasts and reefs varies more than the openp ocean, from day to day and season to season, but you should expect most surface seawater values to  vall between about 7.8 and 8.3. Note that "acidification" means the ocean is becoming **less basic**. it is not actually acidic, and it won't become acidic.

To track it, NOAA keeps buoys moored in the ocean that measure seawater chemistry every few hours. Today you'll compare two of them, over the same winter:

|  |**Kāneʻohe, Hawaiʻi**|**NH10, Oregon**|
|:--|:--|:--|
|where|Kāneʻohe Bay, on the windward coast of Oʻahu, among coral reefs | on the conteintental shelf about 10 nautical miles (18 km) off Newport, Oregon|
|latitude, longitude| 21.48 N, 157.78 W | 44.64 N, 124.30 W|
|file| `Kaneohe_158W_21N_Nov2015_Oct2016.csv`|`NH10_124W_44N_Oct2015_Mar2016.csv`|

*Data: NOAA Ocean Acidification Program moorings, distributed by NOAA NCEI's [Ocean Carbon and Acidification Data System](https://www.ncei.noaa.gov/access/ocean-carbon-acidification-data-system/oceans/Moorings/OAP_Moorings.html); methods in [Sutton et al. (2019)](https://doi.org/10.5194/essd-11-421-2019). Dataset idea from the 2025 EES 3350 final project of Vanderbilt EES student Natalia Freedman.*

### Ocean acidification in a nutshell

When CO$_2$ from the air dissolves in seawater, it reacts with water to make **carbonic acid**, which quickly gives up a hydrogen ion:

$$\text{CO}_2 + \text{H}_2\text{O} \;\rightarrow\; \text{H}_2\text{CO}_3 \;\rightarrow\; \text{H}^+ + \text{HCO}_3^-$$

More hydrogen ions (H$^+$) means lower pH. Some of those extra hydrogen ions then combine with **carbonate ions** (CO$_3^{2-}$) already in the seawater:

$$\text{H}^+ + \text{CO}_3^{2-} \;\rightarrow\; \text{HCO}_3^-$$

That second reaction is the real problem for marine linve. Corals, oysters, clams, and many plankton build their skeletons and shells from **calcium carbonate** (CaCO$_3$), and they need carbonate ions to do it. As the ocean takes up more CO$_2$, carbonate ions get scarcer and building shells takes more energy. When conditions are bad enough, existing shells can start to dissolve (Doney et al., 2009).

Since the industrial revolution, the average pH of the surface of the ocean has dropped from about 8.2 to about 8.1. That sounds small, but pH is a logarithmic scale: it's a 25-30% increase in hydrogen ions. You'll calculate this yourself in Exercise 7.

<center><b>Why these two sites?</b></center>

![map of buoy locations](../figures/lab08_buoy_map.png)

- **NH10, Oregon** (b) sits in a **coastal upwelling** zone. In spring and summer, winds push surface water offshore, and deep water rises to replace it. That deep water is naturally rich in CO$_2$, because it has been collecting CO$_2$ from decomposing organic matter for decades, so upwelling brings low-pH water onto the shelf (Feely et al., 2008). In 2007 and 2008, oyster hatcheries in the Pacific Northwest saw their larvae die off during upwelling events. The cause turned out to be this corrosive water, not disease (Barton et al., 2012). Our record runs from October to March, mostly *outside* upwelling season. Keep that in mind when you interpret it.
- **Kāneʻohe, Hawaiʻi** (a) is ringed by coral reefs. Reefs change the water around them: organisms breath out CO$_2$ at night, algae draw CO$_2$, down during the day, and corals building their skeletons release CO$_2$ as a  by-product:

$$\text{Ca}^{2+} + 2\,\text{HCO}_3^- \;\rightarrow\; \text{CaCO}_3 + \text{CO}_2 + \text{H}_2\text{O}$$

So a reef's water chemistry reflects both the global ocean and the reef's own biology. Offshore of Hawaiʻi, the long-running station ALOHA time series has recorded the open ocean's pH declining steadily since the late 1980s (Dore et al., 2009). In a reef bay, that slow decline happens underneath a much bigger daily and seasonal rhytm. That rhythm is exactly what you'll go looking for in Parts 4 and 5.

<center>Your question today:</center>
<center><b>are these two places experiencing the same ocean?</b></center>

You'll use everything form this week's pandas lectures:
- `read_csv`
- columns and `.loc`
- masks
- new and renamed columns
- dates as the index
- `groupby`

This lab is long on purpose. Start it in lab, and finish it during the week.

## Getting Started

1. Make a copy of this Lab08 document and move it to your Week08 assignment repository.
2. Do all of your work there.
3. Add, Commit, Push whenever you finish a work session and then after you've finished the whole lab.

<div class="alert alert-info">

### Question 0: Import the appropriate libraries.

What libraries do you need to import in order to complete this lab? There should be two of them (at least).
    
</div>

In [ ]:
# your answer here

---
## Part 1: look before you lead
Always look at a raw file before you trust it. Peak at the first 3 lines of each:

In [ ]:
!head -3 Kaneohe_158W_21N_Nov2015_Oct2016.csv

In [ ]:
!head -3 NH10_124W_44N_Oct2015_Mar2016.csv

Each row is one measurement, taken every 3 hours. The columns you'll need:
|**what**|**Kāneʻohe column name**|**NH10 column name**|**units**|
|:--|:--|:--|:--|
|date|`Date`|`Date`| month/day/2-digit year|
|time|`Time`|`Time`| hour:minute, in **UTC** (Coordinate Universal Time|
|seawater pH|`pH SW`|`pH (total scale)`| pH units|
|pH quality flag|`pH QF`|`pH QF`| see part 2|
|sea-surface temperature|`SST (C)`|`SST (C)`|  $^\circ$C|
|salinity|`Salinity`|`Salinity`| practical salinity (no units)|
|CO$_2$ in seawater|`pCO2 SW (sat) uatm`|`pCO2 SW (sat) uatm`| $\mu$atm|
|CO$_2$ in the air above|`pCO2 SW (sat) uatm`|`pCO2 SW (sat) uatm`| $\mu$atm|
|seawater minus air CO$_2$|`dpCO2`|`dpCO2`| $\mu$atm|
|dissolved oxygen|`DOXY`|`DOXY (umol/kg)`| $\mu$mol/kg|

Notice that the **same measurement has different names** in the two files. That's normal for real data, and it's not the first thing we'll fix.

<div class="alert alert-info">

### Exercise 1 : a first look

1. Load each file into a DataFrame, `kan_raw`, and `ore_raw`, with plain `pd.read_csv()`. Print the shape of each.
2. Print all the columns in `kan_raw`. Look cloesly at the names tha tstart with `xCO2`. What's odd about the spacing>
3. Use `.describe()` on **just the pH column** of each file. What are the minimum values? Do they make sense fo rseawater pH?
    
</div>

In [ ]:
# your answer here

<div class="alert alert-warning">

### **Question 1**: If you hadn't looked at `.describe()` and had just computing the average pH of each site, what would you have concluded? Why is it a good habit to check the min and max of every column you use?
    
</div>

> your answer here

---
## Part 2: clean it up

### Fill values --> `NaN`
`pd.read_csv()` can turn a fill value into `NaN` as it read the file, with `na_values=`:

<div class="alert alert-info">

### Exercise 2: load it properly

1. Reload both files as `kan` and `ore`, this time with `nan_values=-999`. Check that the minimum pH now makes sense.
2. How many pH values are **missing** in each file?
3. Each pH measurement also has a **quality flag** in the `pH QF` column. Use `.value_counts()` to see which flags appear in each file, and how often.
    
</div>

In [ ]:
# your answer here

### Quality flags

The flags follow the convention used across oceanography (the WOCE flags), as described in the documentation for these moorings ([Sutton et al., 2019](https://doi.org/10.5194/essd-11-421-2019)):

|**flag**|**meaning**|
:--|:--|
|2|good|
|3|questionable|
|4|bad|
|5|not reported|

A scientist would use only the **good** (flag 2) measurements.

<div class="alert alert-info">

### Exercise 3: same names for different things

1. Make a dictionary for each site that maps its messy column names to the following names. Keep `Date` and `Time` as they are. **Copy the old names from `.columns`.**
    - `pH`
    - `pH_QF`
    - `SST`
    - `SAL`
    - `pCO2_sw`
    - `pCO2_air`
    - `dpCO2`
    - `DO`
2. Use a rename on each DataFrame, keeping **only** `Date`, `Time`, and the 8 clean columns above. Overwrite `kan` and `ore` with this renamed DataFrame.
3. Print the columns for `kan` and `ore`. They should match exactly.
      
    
</div>

In [ ]:
# your answer here

<div class="alert alert-info">

### Exercise 4: dates and times as the index

In these files, the ddate and the time are in **separate** text columns. To make one real datetime for each row:

1. Combine them into one string per row. Print the first 3. (`+` on two text columns sticks them together row by row).
2. Turn that into real datetimes using `format=%m%d%y %H:%M`, store it in a new column `DATETIME`, and make it the index. Do the same for `ore`. You can drop `Date` and `Time` after.
3. Print the **first** and **last** datetime in each record, and the time between the first two measurements.
    
</div>

In [ ]:
# your answer here

<div class="alert alert-info">

### Exercise 5: keep only the good pH

1. Make a mask called `bad_kan` that is `True` where Kāneʻohe's `pH_QF` is **not** 2. Replace those pH values with `NaN`. Do the same for Oregon.
2. Now how many **good** pH values does each site have?
3. Did removing the questionable values change Oregon's average pH? Compute the mean before and after (hint: you'll need to reload or use a copy for "before").
    
</div>

In [ ]:
# your answer here

<div class="alert alert-danger">

### **CHECKPOINT:** commit and push

save your notebook, then:
```
git add lab08_LastName.ipynb
git commit -m "load and clean both buoys"
git push
```
    
</div>

---
## Part 3: same winter, two oceans

To compare fairly, use only the period when **both** buoys were measuring pH: **November 3,  through March 1, 2016**.

<div class="alert alert-info">

### Exercise 6: the overlap

1. Create `kan_w` and `ore_w`, bersions of `kan` and `ore` that only cover Nov 3, 2015 through Mar 1, 2016. Add `.copy()` to the end of each, because you'll add a column to them in Exercise 7, and a copy keeps those changes from touching `kan` and `ore` (views vs. copies).
2. For each site, print the **mean, standard deviation, minimum**, and **maximum** of `pH`, and the mean `SST`. Make it a neat table with f-strings.
3. When was each site's **lowest** pH in this window?
    
</div>

In [ ]:
# your answer here
# 1

# 2
print(f"{'':10s} {'mean':>6s}{'std':>6s} {'min':>6s} {'max':6s} {'SST':>6s}")
for _____ in ____:
    _______
    print(f"{____:10s} {____:6.3f} {____:6.3f} {____:6.3f} {____:6.3f} {____:6.1f}")

# 3

### How much more acidic is "0.12 lower"?
pH is a **logarithmic** scale, like earthquake magnitude in Lab 06:
$$\text{pH} = -\log_{10}[\text{H}^+] \qquad \text{so} \qquad [\text{H}^+] = 10^{-\text{pH}}$$

where $[\text{H}^+]$ is the concentration of hydrogen ions. More hydrogen ions means more acidic.

<div class="alert alert-info">

### Exercise 7: hydrogen ions

1. Add a column `H` to **both** `kan_w` and `ore_w`: the hydrogen-ion concentration, `10 ** (-pH)`, in mol/kg. 
2. How many times more hydrogen ions did Kāneʻohe water have than Oregon's, on average? Express it as a **percent more**.
3. The global surface ocean has lost about 0.1 pH units since pre-industrial times. What percent increase in hydrogen ions is a drop of 0.1?
    
</div>

In [ ]:
# your answer here

---
## Part 4: is the ocean breathing in or breathing out?
The buoys also measure CO$_2$ in the **seatware** and in the **air** right above it. `dpCO2` is the difference:
$$\Delta p\text{CO}_2 = p\text{CO}_{2,\text{seawater}} - p\text{CO}_{2,\text{air}}$$

- if `dpCO2` is **negative**, the water holds less CO$_2$ than the air, so CO$_2$ flows **into** the ocean: the ocean is a **sink**.
- if `dpCO2` is **positive**, CO$_2$ flows **out** of the ocean into the air: the ocean is a **source**.

<div class="alert alert-info">

### Exercise 8: source or sink?

Use each site's **whole* *record (`kan` and `ore`), not just the winter overlap.
1. How many `dpCO2` measurements does each site have? (Not the same as the number of rows!)
2. What **fraction ** of each site's `dpCO2` measurements are **positive**? Divide the count of positive values by the count of **non-missing** values.
3. What is each site's mean `dpCO2`? Which site is a source and which is a sink?
    
</div>

In [ ]:
# your answer here

<div class="alert alert-info">

### Exerccise 9 : Kāneʻohe through the seasons

Kāneʻohe's CO$_2$ record covers March through October 2016.
1. Use `groupby` on the **month** of the index to find Kāneʻohe's mean `dpCO2` for each month.
2. Do the same for mean `SST`.
3. In which month was Kāneʻohe the strongest **sink**? The strongest **source**? (`.idxmin()` and `idxmax()` work on the result of a `groupby`, too.)

</div>

In [ ]:
# your answer here

<div class="alert alert-warning">

### **Question 2**: Warm water can hold less dissolved gas than cold water (that's why a warm soda goes flat faster). Use that idea to explain the pattern you found in Exercise 9.
    
</div>

> your answer here

<div class="alert alert-danger">

### **CHECKPOINT:** commit and push

save your notebook, then:
```
git add lab08_LastName.ipynb
git commit -m "Parts 3 and 4s"
git push
```
    
</div>

---
## Part 5: the daily rhythm

During the day, algae and the tiny algae living inside corals **photosynthesize**: they take CO$_2$ **out** of the water, so pH goes **up** and oxygen goes **up**. At night, photosynthesis stops by everything keeps respiring, adding CO$_2$ back, so pH and oxygen go **down**. If that's happening at Kāneʻohe, we should see it in the hour of the day.

**One catch**: the times in these fiels are in **UTC**. Hawai'i is 10 hours behind UTC, all year (Hawai'i doesn't use daylight savings time). So to get Hawai'i local time, subtract 10 hours. `pd.Timedelta(hours=10)` is a length of time that pandas can add to or subtract from dates.

<div class="alert alert-info">

### Exercise 10: What time is it in Hawai'i?

1. Make `kan_local`, a copy of `kan` whose index is shifted to Hawai'i time (`kan_local.index - kan_local.index - pd.Timedelta(hours=10)`). Print the first 3 index values of `kan` and `kan_local` to check.
2. Use `groupby` on `kan_local.index.hour` to find the **mean pH** at each hour of the day. Some hours come out of `NaN`; add `.dropna()` to the end and remove them. Which hours are left, and why only those?
3. At what **local hour** was pH highest on average, and at what hour was it lowest?
4. Do the same for dissoled oxygen (`DO`). Do pH and oxygen rise and fall together?
    
</div>

In [ ]:
# your answer here

<div class="alert alert-info">

### Exercise 11: How big is the daily swing?

1. For Kāneʻohe, use `groupby` on `kan_local.index.date` to find each day's maximum pH and each day's **minimum pH**. Subtrack to get each day's **range**.
2. What is the **average** daily pH range at Kāneʻohe?
3. Repeat for Oregon (using `ore`: the time zone doesn't matter for a whole-day range, as long as you're consistent). How many times bigger is Kāneʻohe's daily swing?
    
</div>

In [ ]:
# your answer here

<div class="alert alert-warning">

### **Question 3**: Look back at the whole lab. In three or four sentences, answer the opening question: are Kāneʻohe and NH10 experiencing the same ocean? Use at least **three** numbers you calculated.
    
</div>

> your answer here

# End for Undergraduate Students

## Submitting
Save your notebook (ctrl + S), then in the terminal from your Lab 08 repo folder:
```
git status
git add lab08_LastName.ipynb
git commit -m "[insert a message here]"
git push
```
Check your repo's page on GitHub to make sure your commit arrived. You can push as many times as you like before the deadline; your most recent push is what counts.

---
# For Graduate Students Only

<div style="background-color:#e8daef; border:1px solid #d2b4de; color:#6c3483; padding:10px; border-radius:4px; margin-bottom:20px;">


## Exercise G1: Oregon's clock

Oregon **does** use daylight saving time, so "subtract 8 hours" is wrong for part of the record. pandas can handle time zones properly:
```python
ore_local = ore.copy()
ore_local.index = ore_local.index.tz_localize("UTC").tz_convert("US/Pacific")
```
`tz_localize("UTC")` tells pandas the times **are** UTC; `tz_convert()` translates them to Pacific time, including the daylight-saving switch.

1. Make `ore_local` this way. Print the first index value, and the index values on **November 1, 2015**, the day daylight saving time ended. What happens to the UTC offset at the end of each timestamp, and to the local hours of the measurements?
2. Find Oregon's mean pH by local hour, as in Exercise 10 (with `.dropna()`). Compute the size of the "daily cycle": the largest hourly mean minus the smallest. Compare it with Kāneʻohe's.
3. Look closely at the hours in #2. Why do they **alternate** between two groups of values? (Hint: which months land on hours 2, 5, 8, ..., and which land on 1, 4, 7, ...? Compare Oregon's October mean pH with its winter mean.)
4. Redo #2 using only the record **after** daylight saving time ended (`.loc["2015-11-01 02:00":]`). Now how big is Oregon's daily cycle compared with Kāneʻohe's? Suggest two reasons it's so much weaker.

</div>

In [1]:
# your answer here

<div style="background-color:#e8daef; border:1px solid #d2b4de; color:#6c3483; padding:10px; border-radius:4px; margin-bottom:20px;">

## Exercise G2: How tightly are pH and CO₂ linked?

`df[["a", "b"]].corr()` computes the **correlation** between two columns: +1 means they rise and fall together perfectly, −1 means one rises exactly when the other falls, and 0 means no linear relationship.

1. For each site, compute the correlation between `pH` and `pCO2_sw`, and between `pH` and `SST`.
2. Which relationship is stronger, and at which site? Explain the sign of the pH–CO₂ correlation using the chemistry from the introduction.
3. Why might the pH–SST relationship have **different signs** at the two sites? (Think about what changes the water temperature at each place, and what else changes along with it.)

</div>

In [ ]:
# your answer here

# End for Graduate Students
## Submitting
Save your notebook (ctrl + S), then in the terminal from your Lab 08 repo folder:
```
git status
git add lab08_LastName.ipynb
git commit -m "[insert a message here]"
git push
```
Check your repo's page on GitHub to make sure your commit arrived. You can push as many times as you like before the deadline; your most recent push is what counts.

---
## References

- Barton, A., Hales, B., Waldbusser, G. G., Langdon, C., & Feely, R. A. (2012). The Pacific oyster, *Crassostrea gigas*, shows negative correlation to naturally elevated carbon dioxide levels: Implications for near-term ocean acidification effects. *Limnology and Oceanography*, 57(3), 698–710.
- Doney, S. C., Fabry, V. J., Feely, R. A., & Kleypas, J. A. (2009). Ocean acidification: The other CO₂ problem. *Annual Review of Marine Science*, 1, 169–192.
- Dore, J. E., Lukas, R., Sadler, D. W., Church, M. J., & Karl, D. M. (2009). Physical and biogeochemical modulation of ocean acidification in the central North Pacific. *Proceedings of the National Academy of Sciences*, 106(30), 12235–12240.
- Feely, R. A., Sabine, C. L., Hernandez-Ayon, J. M., Ianson, D., & Hales, B. (2008). Evidence for upwelling of corrosive "acidified" water onto the continental shelf. *Science*, 320(5882), 1490–1492.
- Friedlingstein, P., et al. (2023). Global Carbon Budget 2023. *Earth System Science Data*, 15, 5301–5369. https://doi.org/10.5194/essd-15-5301-2023
- IPCC (2021). Summary for Policymakers. In *Climate Change 2021: The Physical Science Basis.* Contribution of Working Group I to the Sixth Assessment Report of the Intergovernmental Panel on Climate Change. Cambridge University Press.
- NOAA National Centers for Environmental Information, Ocean Carbon and Acidification Data System: OAP-funded moorings and time series. [Link](https://www.ncei.noaa.gov/access/ocean-carbon-acidification-data-system/oceans/Moorings/OAP_Moorings.html)
- Sutton, A. J., et al. (2019). Autonomous seawater pCO₂ and pH time series from 40 surface buoys and the emergence of anthropogenic trends. *Earth System Science Data*, 11, 421–439. https://doi.org/10.5194/essd-11-421-2019